# 01 Data Exploration - AG News

In [2]:
#import libraries
import pandas as pd
import numpy as np

## Load dataset

In [5]:
#Load dataset

train_path = "../data/raw/train.csv"
test_path = "../data/raw/test.csv"

train_df = pd.read_csv(train_path)  #read_csv loads the csv file into a pandas dataframe.
test_df = pd.read_csv(test_path)    #a df is like an excel table inside python

train_df.head()         #show the first 5 rows of the training data



,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


## Basic dataset check

In [6]:
train_df.shape   #check the shape of the training data (rows, columns)

(120000, 3)

In [8]:
#show all column names

train_df.columns

Index(['Class Index', 'Title', 'Description'], dtype='str')

In [9]:
#info shows column names, non-null counts, and data types of each column

train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   Class Index  120000 non-null  int64
 1   Title        120000 non-null  str  
 2   Description  120000 non-null  str  
dtypes: int64(1), str(2)
memory usage: 29.7 MB


## Rename columns

In [22]:
# Rename columns to lowercase and snake_case.
# This makes them easier to use in Python.

train_df = train_df.rename(columns={
    "Class Index": "class_index",
    "Title": "title",
    "Description": "description"
})

test_df = test_df.rename(columns={
    "Class Index": "class_index",
    "Title": "title",
    "Description": "description"
})

In [23]:
train_df.columns

Index(['class_index', 'title', 'description'], dtype='str')

## Label Mapping

In [24]:
# mapping numbers to readable calss names

label_map = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

In [ ]:
# create a new column called label_name
# map() replaces each class number with its category name.
train_df["label_name"] = train_df["class_index"].map(label_map)
test_df["label_name"] = test_df["class_index"].map(label_map)

In [27]:
# Show first 5 rows after adding label_name
train_df.head()

,class_index,title,description,label_name
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Business
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Business
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Business
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Business
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...",Business


In [33]:
#count how many articles are in each category
train_df["label_name"].value_counts()

label_name
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64

In [35]:
train_df["label_name"].value_counts()

label_name
Business    30000
Sci/Tech    30000
Sports      30000
World       30000
Name: count, dtype: int64

## Missing value check


In [37]:
#checki missing values in training data
train_df.isna().sum()

class_index    0
title          0
description    0
label_name     0
dtype: int64

In [38]:
# Check missing values in test data
test_df.isna().sum()

class_index    0
title          0
description    0
label_name     0
dtype: int64

## Create Text Column

In [39]:
# fill missing title/description wiht empty string.
# This prevents errors when combinig text.

train_df["title"] = train_df["title"].fillna("")
train_df["description"] = train_df["description"].fillna("")

test_df["title"] = test_df["title"].fillna("")
test_df["description"] = test_df["description"].fillna("")

In [40]:
#combine title and description into one tet column
#This will be the main input text for the ML model later.
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

In [41]:
# Show only important columns to verify the new text column
train_df[["title", "description", "text"]].head()

,title,description,text
0,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
2,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
4,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."


## Light text cleanup

In [42]:
# Replace backslash characters with spaces.

train_df["text"] = train_df["text"].str.replace("\\", " ", regex=False)
test_df["text"] = test_df["text"].str.replace("\\", " ", regex=False)

#Replace newline characters with spaces.
#This keeps each article in one line, which is important for the ML model later.
train_df["text"] = train_df["text"].str.replace("\n", " ", regex=False)
test_df["text"] = test_df["text"].str.replace("\n", " ", regex=False)

#Remove extra spaces from the start and end of text
train_df["text"] = train_df["text"].str.strip()
test_df["text"] = test_df["text"].str.strip()

In [43]:
# Check cleaned text
train_df["text"].head()

0    Wall St. Bears Claw Back Into the Black (Reute...
1    Carlyle Looks Toward Commercial Aerospace (Reu...
2    Oil and Economy Cloud Stocks' Outlook (Reuters...
3    Iraq Halts Oil Exports from Main Southern Pipe...
4    Oil prices soar to all-time record, posing new...
Name: text, dtype: str

## Article length analysis

In [46]:
# char_count counts the total number of characters in each article.
# It includes letters, spaces, punctuation, and numbers.
train_df["char_count"] = train_df["text"].apply(len)

# word_count counts how many words are in each article.
# split() breaks text into words using spaces.
train_df["word_count"] = train_df["text"].apply(lambda text: len(text.split()))

In [47]:
# describ() gives useful statistics:
# count = number of rows
# mean = average number of characters/words
# min = shortest value
# max = longest value
# 25%, 50%, 75% = distribution points
train_df[["char_count", "word_count"]].describe()

,char_count,word_count
count,120000.000000,120000.000000
mean,236.333000,37.967250
std,66.509242,10.179913
min,17.000000,4.000000
25%,196.000000,32.000000
50%,232.000000,37.000000
75%,266.000000,43.000000
max,1012.000000,177.000000


## Longest Article

In [48]:
#idxmax() finds the row index where char_count is largest
longest_index = train_df["char_count"].idxmax()

#loc[] selects that row and show only useful columns.
train_df.loc[
    longest_index,
    ["label_name", "title", "char_count", "word_count", "text"]
]

label_name                                             Sci/Tech
title          Yahoo, SBC extend partnership, plan new services
char_count                                                 1012
word_count                                                  109
text          Yahoo, SBC extend partnership, plan new servic...
Name: 95659, dtype: object

## Shortest Article

In [49]:
# idxmin() finds the row index where char_count is smallest
shortest_index =train_df["char_count"].idxmin()

#s show the shortest article details
train_df.loc[
    shortest_index,
    ["label_name", "title", "char_count", "word_count", "text"]
]


label_name               Sports
title                Top of 3rd
char_count                   17
word_count                    4
text          Top of 3rd #NAME?
Name: 22955, dtype: object

## Sample Articles by Category

In [57]:
#loop through each category name
for label in train_df["label_name"].unique():
    print("category:", label)

    #select first 5 rows from the current category
    samples = train_df[train_df["label_name"] == label].head(5)

    #print only first 300 characters so output is readable
    for text in samples["text"]:
        print(text[:300])

    print("...")

category: Business
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling band of ultra-cynics, are seeing green again.
Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group, which has a reputation for making well-timed and occasionally controversial plays in the defense industry, has quietly placed its bets on another part of the market.
Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries about the economy and the outlook for earnings are expected to hang over the stock market next week during the depth of the summer doldrums.
Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export flows from the main pipeline in southern Iraq after intelligence showed a rebel militia could strike infrastructure, an oil official said on Saturday.
Oil prices soar to all-time record, posing new menace to US economy (AFP) AFP -

## Save Small Sample CSV

In [58]:
# sample() randomly selects 1000 rows from training data.
# random_state=42 makes the random sample reproducible.
sample_df = train_df.sample(1000, random_state=42)

In [59]:
# Keep only useful columns for the sample file.
sample_df = sample_df[
    ["class_index", "label_name", "title", "description", "text", "char_count", "word_count"]
]

In [60]:
#save the sample csv inside data/.
#index=False prevents pandas from adding a new index column to the csv file.
sample_df.to_csv("../data/sample_news.csv", index=False)

In [61]:
# Check first 5 rows of the saved sample DataFrame.
sample_df.head()

,class_index,label_name,title,description,text,char_count,word_count
71787,3,Business,"BBC set for major shake-up, claims newspaper","London - The British Broadcasting Corporation,...","BBC set for major shake-up, claims newspaper L...",288,46
67218,3,Business,Marsh averts cash crunch,Embattled insurance broker #39;s banks agree t...,Marsh averts cash crunch Embattled insurance b...,174,28
54066,2,Sports,"Jeter, Yankees Look to Take Control (AP)",AP - Derek Jeter turned a season that started ...,"Jeter, Yankees Look to Take Control (AP) AP - ...",165,30
7168,4,Sci/Tech,Flying the Sun to Safety,When the Genesis capsule comes back to Earth w...,Flying the Sun to Safety When the Genesis caps...,173,34
29618,3,Business,Stocks Seen Flat as Nortel and Oil Weigh,NEW YORK (Reuters) - U.S. stocks were set to ...,Stocks Seen Flat as Nortel and Oil Weigh NEW ...,385,45


## Day 1 Summary

Today I loaded the Kaggle AG News dataset and explored its basic structure. The dataset has class index, title, and description columns. I renamed the columns, mapped class numbers to readable category names, combined title and description into one text field, performed light cleanup, checked missing values, calculated article lengths, viewed examples from each category, and saved a small sample CSV for future testing.